In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- expose_trx_join_filter ---
FIX_EXPOSE_TRX_JOIN_FILTER_DATE_COLS = ["pol_date_yr", "pol_date_end"]
FIX_EXPOSE_TRX_JOIN_FILTER_DATE_LOOKUP_PD = pd.DataFrame({"pol_num":["P1","P2"],"pol_date_yr":pd.to_datetime(["2020-01-01","2021-01-01"]),"pol_date_end":pd.to_datetime(["2020-12-31","2021-12-31"])})
FIX_EXPOSE_TRX_JOIN_FILTER_TRX_DATA_PD = pd.DataFrame({"pol_num":["P1","P2"],"trx_date":pd.to_datetime(["2020-06-01","2021-06-01"]),"trx_type":["A","B"],"trx_amt":[100.,200.]})
FIX_EXPOSE_TRX_JOIN_FILTER_DATE_LOOKUP_PL = pl.from_pandas(FIX_EXPOSE_TRX_JOIN_FILTER_DATE_LOOKUP_PD)
FIX_EXPOSE_TRX_JOIN_FILTER_TRX_DATA_PL = pl.from_pandas(FIX_EXPOSE_TRX_JOIN_FILTER_TRX_DATA_PD)
FIX_EXPOSE_TRX_JOIN_FILTER_DATE_LOOKUP = FIX_EXPOSE_TRX_JOIN_FILTER_DATE_LOOKUP_PD
FIX_EXPOSE_TRX_JOIN_FILTER_TRX_DATA = FIX_EXPOSE_TRX_JOIN_FILTER_TRX_DATA_PD
FIX_EXPOSE_TRX_JOIN_FILTER_TRX_DATE = "2020-06-01"

# --- expose_trx_left_join_fill ---
FIX_EXPOSE_TRX_LEFT_JOIN_FILL_DATE_COLS = ["pol_date_yr", "pol_date_end"]
FIX_EXPOSE_TRX_LEFT_JOIN_FILL_TRX_DATA_BEFORE = pd.DataFrame({
    "pol_num":[1,2],
    "pol_date_yr":pd.to_datetime(["2020-01-01","2021-01-01"]),
    "trx_amt_claim":[100.,200.],
    "trx_n_claim":[1,1],
})
FIX_EXPOSE_TRX_LEFT_JOIN_FILL_TRX_DATA_GEN = pl.from_pandas(FIX_EXPOSE_TRX_LEFT_JOIN_FILL_TRX_DATA_BEFORE)

# --- expose_trx_pivot ---
FIX_EXPOSE_TRX_PIVOT_COLUMN = "trx_type"
FIX_EXPOSE_TRX_PIVOT_DATE_COLS = ["pol_date_yr", "pol_date_end"]
FIX_EXPOSE_TRX_PIVOT_FLATTEN = True
FIX_EXPOSE_TRX_PIVOT_PIVOT_TABLE = None
FIX_EXPOSE_TRX_PIVOT_TRX_DATA_PD = pd.DataFrame({"pol_num":["P1","P2"],"pol_date_yr":pd.to_datetime(["2020-01-01","2021-01-01"]),"trx_date":pd.to_datetime(["2020-06-01","2021-06-01"]),"trx_type":["A","B"],"trx_n":[1,1],"trx_amt":[100.,200.]})
FIX_EXPOSE_TRX_PIVOT_TRX_DATA_PL = pl.from_pandas(FIX_EXPOSE_TRX_PIVOT_TRX_DATA_PD)
FIX_EXPOSE_TRX_PIVOT_TRX_DATA = FIX_EXPOSE_TRX_PIVOT_TRX_DATA_PD

# --- expose_trx_unique ---
FIX_EXPOSE_TRX_UNIQUE_DATE_COLS = ["pol_date_yr", "pol_date_end"]
FIX_EXPOSE_TRX_UNIQUE_DATE_LOOKUP_BEFORE = pd.DataFrame({"pol_num":[1,2],"pol_date_yr":pd.to_datetime(["2020-01-01","2021-01-01"]),"pol_date_end":pd.to_datetime(["2020-12-31","2021-12-31"])} )
FIX_EXPOSE_TRX_UNIQUE_DATE_LOOKUP_GEN = pl.from_pandas(FIX_EXPOSE_TRX_UNIQUE_DATE_LOOKUP_BEFORE)
FIX_EXPOSE_TRX_UNIQUE_TRX_COLS = ["trx_n_A","trx_amt_A"]
FIX_EXPOSE_TRX_UNIQUE_TRX_DATA_BEFORE = pd.DataFrame({
    "pol_num":[1,2],
    "trx_date":pd.to_datetime(["2020-06-01","2021-06-01"]),
    "trx_type":["A","A"],
    "trx_amt":[100.,200.],
})
FIX_EXPOSE_TRX_UNIQUE_TRX_DATA_GEN = pl.from_pandas(FIX_EXPOSE_TRX_UNIQUE_TRX_DATA_BEFORE)

print("✅ Fixtures loaded")
_ACTXPS_SELF_DATA_BEFORE = pd.DataFrame({
    "pol_num":[1,2,3],
    "pol_date_yr":pd.to_datetime(["2020-01-01","2021-01-01","2022-01-01"]),
    "pol_date_end":pd.to_datetime(["2020-12-31","2021-12-31","2022-12-31"]),
    "trx_amt_claim":[100.0,200.0,0.0],
    "trx_n_claim":[1,2,0],
})
_ACTXPS_SELF_DATA_GEN = pl.from_pandas(_ACTXPS_SELF_DATA_BEFORE)
self = SimpleNamespace(data=_ACTXPS_SELF_DATA_BEFORE.copy(), trx_types=["claim"])



In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_expose_trx_join_filter(date_cols, date_lookup, trx_data, trx_date):
    trx_data = (trx_data
                .merge(date_lookup, how='inner', on='pol_num')
                .query(f"(trx_date >= {date_cols[0]}) & (trx_date <= {date_cols[1]})"))
    trx_data['trx_n'] = 1
    return trx_data

def before_expose_trx_left_join_fill(date_cols, trx_data):
    self.data = (self.data.
                 merge(trx_data,
                       on=['pol_num', date_cols[0]],
                       how='left'))
    trx_cols = [x for x in self.data.columns if x.startswith('trx_')]
    self.data.loc[:, trx_cols] = \
        self.data.loc[:, trx_cols].apply(lambda x: x.fillna(0))
    return trx_cols

def before_expose_trx_pivot(column, date_cols, flatten, pivot_table, trx_data):
    trx_data = (trx_data
                .pivot_table(values=['trx_n', 'trx_amt'],
                             index=['pol_num', date_cols[0]],
                             columns='trx_type',
                             aggfunc='sum',
                             observed=True,
                             fill_value=0)
                .reset_index())
    if flatten:
        cols = trx_data.columns.to_flat_index()
        cols = ['_'.join(x) if isinstance(x, tuple) and x[1] != '' else (x[0] if isinstance(x, tuple) else x) for x in cols]
        trx_data.columns = cols
    return trx_data

def before_expose_trx_unique(date_cols, date_lookup, trx_cols, trx_data=None):
    if trx_data is None:
        trx_data = pd.DataFrame({"trx_type":["rider_death"],"trx_n":[1],"trx_amt":[100.0],"pol_num":[1],"period_end":[1]})
    new_trx_types = pd.unique(trx_data.trx_type)
    existing_trx_types = self.trx_types
    conflict_trx_types = np.intersect1d(new_trx_types, existing_trx_types)

    trx_data = (trx_data
                .merge(date_lookup, how='inner', on='pol_num')
                .query(f"(trx_date >= {date_cols[0]}) & (trx_date <= {date_cols[1]})"))

    trx_data['trx_n'] = 1
    trx_data = (trx_data
                .pivot_table(values=['trx_n', 'trx_amt'],
                             index=['pol_num', date_cols[0]],
                             columns='trx_type',
                             aggfunc='sum',
                             observed=True,
                             fill_value=0)
                .reset_index())

    cols = trx_data.columns.to_flat_index()
    cols = ['_'.join(x) if x[1] != '' else x[0] for x in cols]
    trx_data.columns = cols

    self.data = self.data.merge(trx_data, on=['pol_num', date_cols[0]], how='left')
    self.data.loc[:, trx_cols] = self.data.loc[:, trx_cols].apply(lambda x: x.fillna(0))
    return cols

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_expose_trx_join_filter(date_cols, date_lookup, trx_data, trx_date):
    trx_data = (
        trx_data.join(date_lookup, how='inner', on='pol_num')
        .filter(
            (pl.col('trx_date') >= pl.lit(date_cols[0])) &
            (pl.col('trx_date') <= pl.lit(date_cols[1]))
        )
    )
    trx_data = trx_data.with_columns(pl.lit(1).alias('trx_n'))
    return trx_data

def gen_expose_trx_left_join_fill(date_cols, trx_data):
    self.data = self.data.join(
        trx_data,
        on=['pol_num', date_cols[0]],
        how='left',
    )
    trx_cols = [x for x in self.data.columns if x.startswith('trx_')]
    self.data = self.data.with_columns([pl.col(x).fill_null(0) for x in trx_cols])
    return trx_cols

def gen_expose_trx_pivot(column, date_cols, flatten, pivot_table, trx_data):
    # ❌ SyntaxError: unmatched ')' (line 8)
    raise NotImplementedError("snippet has unfixable syntax")

def gen_expose_trx_unique(date_cols, date_lookup, trx_cols, trx_data=None):
    if trx_data is None:
        trx_data = pl.DataFrame({"trx_type":["rider_death"],"trx_n":[1],"trx_amt":[100.0],"pol_num":[1],"period_end":[1]})
    new_trx_types = trx_data.get_column("trx_type").unique().to_list()
    existing_trx_types = self.trx_types
    conflict_trx_types = np.intersect1d(new_trx_types, existing_trx_types)

    trx_data = (trx_data
                .join(date_lookup, how='inner', on='pol_num')
                .filter((pl.col("trx_date") >= pl.col(date_cols[0])) & (pl.col("trx_date") <= pl.col(date_cols[1]))))

    trx_data = trx_data.with_columns(pl.lit(1).alias('trx_n'))

    trx_n_data = (trx_data
                  .pivot(index=['pol_num', date_cols[0]],
                         columns='trx_type',
                         values='trx_n',
                         aggregate_function='sum'))
    trx_amt_data = (trx_data
                    .pivot(index=['pol_num', date_cols[0]],
                           columns='trx_type',
                           values='trx_amt',
                           aggregate_function='sum'))

    trx_n_cols = [c for c in trx_n_data.columns if c not in ['pol_num', date_cols[0]]]
    trx_amt_cols = [c for c in trx_amt_data.columns if c not in ['pol_num', date_cols[0]]]

    trx_n_data = trx_n_data.rename({c: f'trx_n_{c}' for c in trx_n_cols})
    trx_amt_data = trx_amt_data.rename({c: f'trx_amt_{c}' for c in trx_amt_cols})

    trx_data = trx_n_data.join(trx_amt_data, on=['pol_num', date_cols[0]], how='inner')

    self.data = self.data.join(trx_data, on=['pol_num', date_cols[0]], how='left')
    self.data = self.data.with_columns([pl.col(c).fill_null(0) for c in trx_cols])
    return trx_data

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: expose_trx_left_join_fill ===

# L1 smoke – generated
try:
    self.data = _ACTXPS_SELF_DATA_GEN.select(["pol_num", "pol_date_yr", "pol_date_end"])
    _r = gen_expose_trx_left_join_fill(FIX_EXPOSE_TRX_LEFT_JOIN_FILL_DATE_COLS, FIX_EXPOSE_TRX_LEFT_JOIN_FILL_TRX_DATA_GEN)
    print("✅ L1 smoke gen_expose_trx_left_join_fill: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_expose_trx_left_join_fill: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    self.data = _ACTXPS_SELF_DATA_BEFORE.loc[:, ["pol_num", "pol_date_yr", "pol_date_end"]].copy()
    _rb = before_expose_trx_left_join_fill(FIX_EXPOSE_TRX_LEFT_JOIN_FILL_DATE_COLS, FIX_EXPOSE_TRX_LEFT_JOIN_FILL_TRX_DATA_BEFORE.copy())
    print("✅ L1 smoke before_expose_trx_left_join_fill: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_expose_trx_left_join_fill: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence – compare side-effect on self.data and returned trx column list.
try:
    self.data = _ACTXPS_SELF_DATA_BEFORE.loc[:, ["pol_num", "pol_date_yr", "pol_date_end"]].copy()
    _rb_cols = before_expose_trx_left_join_fill(FIX_EXPOSE_TRX_LEFT_JOIN_FILL_DATE_COLS, FIX_EXPOSE_TRX_LEFT_JOIN_FILL_TRX_DATA_BEFORE.copy())
    _before_data = self.data.copy()
    self.data = _ACTXPS_SELF_DATA_GEN.select(["pol_num", "pol_date_yr", "pol_date_end"])
    _rg_cols = gen_expose_trx_left_join_fill(FIX_EXPOSE_TRX_LEFT_JOIN_FILL_DATE_COLS, FIX_EXPOSE_TRX_LEFT_JOIN_FILL_TRX_DATA_GEN)
    _gen_data = self.data.clone()
    compare(_before_data, _gen_data, "expose_trx_left_join_fill self.data")
    if list(_rb_cols) == list(_rg_cols):
        print("✅ L2 equivalence expose_trx_left_join_fill trx_cols: MATCH")
    else:
        print(f"❌ L2 equivalence expose_trx_left_join_fill trx_cols: MISMATCH — before={_rb_cols}, gen={_rg_cols}")
except Exception as _e:
    print(f"❌ L2 equivalence expose_trx_left_join_fill: setup error — {type(_e).__name__}: {_e}")

# L3 edge - empty frames; compare return value and self.data mutation.
try:
    self.data = _ACTXPS_SELF_DATA_BEFORE.loc[:, ["pol_num", "pol_date_yr", "pol_date_end"]].head(0).copy()
    _rb_cols = before_expose_trx_left_join_fill(
        FIX_EXPOSE_TRX_LEFT_JOIN_FILL_DATE_COLS,
        FIX_EXPOSE_TRX_LEFT_JOIN_FILL_TRX_DATA_BEFORE.head(0).copy(),
    )
    _before_data = self.data.copy()
    self.data = _ACTXPS_SELF_DATA_GEN.select(["pol_num", "pol_date_yr", "pol_date_end"]).head(0)
    _rg_cols = gen_expose_trx_left_join_fill(
        FIX_EXPOSE_TRX_LEFT_JOIN_FILL_DATE_COLS,
        FIX_EXPOSE_TRX_LEFT_JOIN_FILL_TRX_DATA_GEN.head(0),
    )
    _gen_data = self.data.clone()
    compare(_before_data, _gen_data, "L3 edge expose_trx_left_join_fill empty self.data", check_row_order=True)
    if list(_rb_cols) == list(_rg_cols):
        print("✅ L3 edge expose_trx_left_join_fill empty trx_cols: MATCH")
    else:
        print(f"❌ L3 edge expose_trx_left_join_fill empty trx_cols: MISMATCH — before={_rb_cols!r}, gen={_rg_cols!r}")
except Exception as _e: print(f"❌ L3 edge expose_trx_left_join_fill: {type(_e).__name__}: {_e}")
